### NanoGPT Implementation
- Implementation of Karpathy's lecture on building GPT from scratch: https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=10

- Google collab from lecture: https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=O6medjfRsLD9


### Get input data

In [ ]:
# import urllib.request

# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# urllib.request.urlretrieve(url, "input.txt")

('input.txt', <http.client.HTTPMessage at 0x108589090>)

### Process data

In [ ]:
# read it in to inspect it
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [4]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [6]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [12]:
# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

print("".join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [28]:
# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}

# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

print(encode("my name is zeal"))
print(decode(encode("my name is zeal")))

[51, 63, 1, 52, 39, 51, 43, 1, 47, 57, 1, 64, 43, 39, 50]
my name is zeal


In [ ]:
# train and test splits
import torch

data = torch.tensor(encode(text), dtype=torch.long)

# 90% training and 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Data size: Train = {len(train_data)} | Val = {len(val_data)}")

Data size: Train = 1003854 | Val = 111540


In [ ]:
# extract one block of training examples
# a single block contains block_size number of examples

block_size = 8  # same as context_length

x = train_data[:block_size]  # single block
y = train_data[1 : block_size + 1]

for i in range(block_size):
    context = x[: i + 1]
    target = y[i]
    print(f"Example {i} --> context = {context} and target = {target} ")


Example 0 --> context = tensor([18]) and target = 47 
Example 1 --> context = tensor([18, 47]) and target = 56 
Example 2 --> context = tensor([18, 47, 56]) and target = 57 
Example 3 --> context = tensor([18, 47, 56, 57]) and target = 58 
Example 4 --> context = tensor([18, 47, 56, 57, 58]) and target = 1 
Example 5 --> context = tensor([18, 47, 56, 57, 58,  1]) and target = 15 
Example 6 --> context = tensor([18, 47, 56, 57, 58,  1, 15]) and target = 47 
Example 7 --> context = tensor([18, 47, 56, 57, 58,  1, 15, 47]) and target = 58 


In [40]:
# data loader

block_size = 8
batch_size = 4


def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    return x, y


x, y = get_batch(split="train")

print(f"Inputs: {x.shape} \n {x}")
print(f"Targets: {y.shape} \n {y}")


Inputs: torch.Size([4, 8]) 
 tensor([[ 1, 30, 21, 15, 20, 13, 30, 16],
        [44, 56, 43, 43,  2,  0,  0, 17],
        [58, 46, 53, 57, 43,  1, 54, 53],
        [43, 57,  1, 45, 50, 43, 39, 52]])
Targets: torch.Size([4, 8]) 
 tensor([[30, 21, 15, 20, 13, 30, 16,  1],
        [56, 43, 43,  2,  0,  0, 17, 24],
        [46, 53, 57, 43,  1, 54, 53, 61],
        [57,  1, 45, 50, 43, 39, 52,  1]])
